# 3.2 The training sweep

**What this notebook is for.** Understanding, sizing and launching the sweep. A
sweep is the cross product of: model, feature-list length, preprocessing recipe,
and model hyperparameters. It gets large fast, and the first job of this notebook
is to make you look at *how* large before you start it.

**Arms and configs.** An **arm** is one `(model, top_n, preprocessing_id)`
triple — the unit of parallelism and the unit of resumption. A **config** is one
arm plus one hyperparameter point. `sweep_arms` enumerates arms with `start` and
`count` so a sweep can be sharded across machines; `run_arm` runs every config in
one arm; `run_sweep` runs a list of arms.

**Everything is resumable.** `run_arm` writes a `_READY` marker per arm and
`run_config` checks for one per config, so re-running a sweep after a failure
skips the completed work. `skip_ready_arms=True` is the default and you almost
never want `force=True` — it exists for the case where the *code* changed and the
old results are wrong, which is a decision to make deliberately.

**`max_minutes_per_arm` is a guard, not a target.** An arm that exceeds it is
recorded as timed out and the sweep continues. Without it one pathological
hyperparameter point — an autoencoder with too many epochs, HDBSCAN on too many
rows — takes the whole sweep down with it.

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# The notebooks live two levels below the repo root. Put the root on sys.path so
# `import trust_score_05` resolves without an editable install, which is what
# happens on EMR and SageMaker where the package arrives as a zip.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("repo root:", REPO_ROOT)

In [ ]:
from trust_score_05.ml.config import DEFAULT_PREPROCESSING_IDS, load_ml_config

ENV = "dev"
cfg = load_ml_config(env=ENV)

print("models          :", cfg.models)
print("top n features  :", cfg.top_n_features)
print("preprocessing   :", DEFAULT_PREPROCESSING_IDS)
print("scenarios       :", cfg.scenarios)
print("levels          :", cfg.aggregation_levels)
print("seed            :", cfg.seed)

## How big is this?

`grid_shape` reports the hyperparameter grid size per model without building it.
Multiply through before launching. The grid dimensions are in `GRID_DIMENSIONS`,
and the fact that `score_variant` was once missing from that tuple — so the
k-means score variants were never actually swept — is a finding in the ML
document; the lesson is that a grid you have not counted is a grid you do not
know the shape of.

In [ ]:
from trust_score_05.ml.grids import GRID_DIMENSIONS, grid_shape
from trust_score_05.ml.sweep import sweep_arms

print("grid dimensions:", GRID_DIMENSIONS)
print()

N_FEATURES = max(cfg.top_n_features)
N_ROWS = cfg.max_training_rows

total_configs = 0
for model in cfg.models:
    per_model = grid_shape(model, cfg, n_features=N_FEATURES, n_rows=N_ROWS)
    arms = len(cfg.top_n_features) * len(DEFAULT_PREPROCESSING_IDS)
    total_configs += per_model * arms
    print(f"{model:22s} grid={per_model:4d}  arms={arms:3d}  configs={per_model * arms:5d}")

print()
print(f"total configs: {total_configs}")
print(f"scenarios    : {len(cfg.scenarios)}  ->  {total_configs * len(cfg.scenarios)} scorings")

## The arms

`sweep_arms(cfg, models, method, start, count)` is the sharding interface. On one
machine, leave `start` and `count` alone. Across N machines, machine `i` takes
`start=i * chunk, count=chunk`. The enumeration order is stable, which is what
makes that safe.

In [ ]:
from trust_score_05.ml.selection import METHOD_NAME

arms = sweep_arms(cfg, models=cfg.models, method=METHOD_NAME)
print(f"{len(arms)} arm(s)")
print()
for arm in arms[:12]:
    print(f"  {arm.model:22s} top_n={arm.top_n_features:4d}  prep={arm.preprocessing_id}")
if len(arms) > 12:
    print(f"  ... and {len(arms) - 12} more")

## Run one arm first

Always. An arm is minutes; a sweep is hours. `run_arm` returns a `RunOutcome`
carrying a `ConfigOutcome` per config, each with a status from `CONFIG_STATUSES`
— so a partial success is legible rather than being one exception.

In [ ]:
import functools

from trust_score_05.ml.datasets import load_fraud_frame, load_scenario_frame, load_training_matrix
from trust_score_05.ml.jobs.base import build_spark_session

spark = build_spark_session("sweep-notebook")
spark.sparkContext.setLogLevel("WARN")

arm = arms[0]
print("arm:", arm)

# The loaders are passed in rather than called, so run_arm controls when the
# read happens and can skip it entirely for an arm that is already ready.
def training_loader(features):
    return load_training_matrix(spark, cfg, features=features)

fraud_frame = load_fraud_frame(spark, cfg, features=None)

def scenario_loader(scenario, features):
    return load_scenario_frame(
        spark, cfg, scenario=scenario, fraud_frame=fraud_frame, features=features
    )

In [ ]:
from trust_score_05.ml.sweep import run_arm

RUN = False  # flip to True to actually train

if RUN:
    outcome = run_arm(
        cfg,
        arm,
        features=None,  # None -> read the selected list for this arm
        train_frame=training_loader(None),
        scenarios=cfg.scenarios,
        scenario_loader=scenario_loader,
        max_minutes=cfg.max_minutes_per_arm,
    )
    print("status   :", outcome.status)
    print("configs  :", len(outcome.configs))
    for config in outcome.configs[:10]:
        print(f"  [{config.status:10s}] {config.hyperparam_id}")
else:
    print("RUN is False -- nothing trained.")
    outcome = None

## Collect the metrics

`collect_metrics` flattens a list of outcomes into one long DataFrame. This is
what the leaderboard in the next notebook reads, and it is written to
`sweep_summary_prefix(cfg)` by the sweep job.

In [ ]:
from trust_score_05.ml.sweep import collect_metrics

if outcome is not None:
    summary = collect_metrics([outcome])
    print("shape:", summary.shape)
    display(summary.head(20))
else:
    print("no outcome to collect")

## The full sweep

Do not run this in a notebook. It is here so that the arguments are documented in
the same place as everything else, and so you can read what the job does. The
real invocation is:

```
python -m trust_score_05.ml.jobs.sweep_job --env dev \
    --models isolation_forest,ecod,copod \
    --start 0 --count 12
```

`--start` and `--count` shard the arms. Run several of those in parallel on
separate machines; the `_READY` markers keep them from duplicating work even if
the shards overlap.

In [ ]:
from trust_score_05.ml.sweep import run_sweep

FULL_SWEEP = False

if FULL_SWEEP:
    outcomes = run_sweep(
        cfg,
        arms=arms,
        feature_loader=None,
        training_loader=training_loader,
        scenarios=cfg.scenarios,
        scenario_loader=scenario_loader,
        max_minutes_per_arm=cfg.max_minutes_per_arm,
        skip_ready_arms=True,
    )
    summary = collect_metrics(outcomes)
    print("shape:", summary.shape)
else:
    print("FULL_SWEEP is False. Use the sweep_job CLI for a real sweep.")

## What next

`3.3-evaluate-and-compare.ipynb` reads the summary and builds the leaderboard.
`3.4-drift.ipynb` checks whether the winning model's score distribution has
moved since it was fitted.